# 🎬 AI Scene Generator - AnimateDiff (Fixed)

Generate full animated scenes with character movement, emotions, and lip-sync.

**Instructions:**
1. Click **Runtime → Run all**
2. Upload your character image
3. Upload your audio file
4. Enter a description prompt
5. Wait for generation (10-30 minutes)
6. Download the result

**Free GPU:** This uses Google Colab's free GPU. Processing takes time but costs nothing!

In [ ]:
# Install dependencies with compatible versions
!pip install -q torch==2.1.0 torchvision==0.16.0
!pip install -q diffusers==0.25.0 transformers accelerate xformers einops omegaconf safetensors
!pip install -q imageio imageio-ffmpeg moviepy
print("✓ Dependencies installed")

In [ ]:
# Download AnimateDiff models (updated paths)
from huggingface_hub import hf_hub_download
import os

os.makedirs("models", exist_ok=True)

print("Downloading AnimateDiff motion module...")
try:
    # Try new repository
    motion_module = hf_hub_download(
        repo_id="guoyww/animatediff",
        filename="mm_sd_v15_v2.ckpt",
        local_dir="models"
    )
except:
    # Fallback to alternative
    print("Using alternative model source...")
    !wget -q -O models/mm_sd_v15_v2.ckpt https://huggingface.co/guoyww/animatediff/resolve/main/mm_sd_v15_v2.ckpt

print("✓ Models downloaded")

In [ ]:
# Upload files
from google.colab import files
import shutil

print("📸 Upload your CHARACTER IMAGE:")
uploaded = files.upload()
character_image = list(uploaded.keys())[0]
print(f"✓ Character image: {character_image}")

print("\n🎵 Upload your AUDIO FILE:")
uploaded = files.upload()
audio_file = list(uploaded.keys())[0]
print(f"✓ Audio file: {audio_file}")

In [ ]:
# Get user prompt
prompt = input("\n✍️ Describe your scene (e.g., 'animated skeleton character speaking with emotions, dramatic lighting, full body'): ")
print(f"\nPrompt: {prompt}")

In [ ]:
# Generate video with AnimateDiff
import torch
from diffusers import AnimateDiffPipeline, DDIMScheduler, MotionAdapter
from diffusers.utils import export_to_video
from PIL import Image
import numpy as np

print("🎬 Initializing AnimateDiff...")

# Load motion adapter
adapter = MotionAdapter.from_pretrained("guoyww/animatediff-motion-adapter-v1-5-2", torch_dtype=torch.float16)

# Create model ID
model_id = "SG161222/Realistic_Vision_V5.1_noVAE"

# Load pipeline
pipe = AnimateDiffPipeline.from_pretrained(
    model_id,
    motion_adapter=adapter,
    torch_dtype=torch.float16
)
scheduler = DDIMScheduler.from_pretrained(
    model_id,
    subfolder="scheduler",
    clip_sample=False,
    timestep_spacing="linspace",
    beta_schedule="linear",
    steps_offset=1,
)
pipe.scheduler = scheduler

# Enable memory optimizations
pipe.enable_vae_slicing()
pipe.enable_model_cpu_offload()

print("✓ Pipeline ready")

# Load character image as reference
character = Image.open(character_image).convert("RGB")
character = character.resize((512, 512))

print("\n🎨 Generating animated scene...")
print("This will take 10-30 minutes. Please be patient!")

# Generate with better settings
output = pipe(
    prompt=prompt,
    negative_prompt="blurry, bad quality, distorted, ugly",
    num_frames=16,
    guidance_scale=7.5,
    num_inference_steps=20,
    generator=torch.Generator("cpu").manual_seed(42)
)

frames = output.frames[0]
export_to_video(frames, "animated_scene.mp4", fps=8)

print("✓ Animation generated!")

In [ ]:
# Add audio to video
from moviepy.editor import VideoFileClip, AudioFileClip

print("🎵 Adding audio to video...")

video = VideoFileClip("animated_scene.mp4")
audio = AudioFileClip(audio_file)

# Loop video to match audio length if needed
if audio.duration > video.duration:
    # Calculate how many times to loop
    loops = int(audio.duration / video.duration) + 1
    video = video.loop(n=loops).subclip(0, audio.duration)

final_video = video.set_audio(audio)
final_video.write_videofile("final_scene.mp4", codec="libx264", audio_codec="aac", logger=None)

video.close()
audio.close()

print("✓ Final video ready!")

In [ ]:
# Download result
from google.colab import files

print("📥 Downloading your animated scene...")
files.download("final_scene.mp4")

print("\n✅ DONE! Your AI-generated scene is ready!")
print("\nYou can generate more scenes by running this notebook again.")